# Simple PID loop double check
Craig Lage - 20Mar26

In [ ]:
import numpy as np
import pickle as pkl
import pandas as pd
import matplotlib.pyplot as plt
from lsst.ts.ofc.state_estimator import StateEstimator
from lsst.ts.ofc.ofc import OFC
from lsst.ts.ofc import OFCData


## Get the Nightly report data and the PID logs.

In [ ]:
day_obs = 20260326

# Load the nightly table.
# The parquet file was created with: 
# https://github.com/lsst-sitcom/ts_aos_analysis/blob/tickets/DM-54406/notebooks/nightly_report/nightly_report_ts_version.ipynb
parquet_file = f"/home/c/cslage/u/MTAOS/times_square_reports/nightly_aos_table_{day_obs}.parquet"
table = pd.read_parquet(parquet_file)
print(f'Loaded {parquet_file}: {len(table)} rows')
print(f'Columns: {sorted(table.columns.tolist())}')

# Load the PID logs
# This was created with PID_Corrections_19Mar26.ipynb
filename = f"/home/c/cslage/u/MTAOS/times_square_reports/PID_data_{day_obs}.pkl"
with open(filename, 'rb') as f:
    pidDict = pkl.load(f)
    

## Next I try to calculate the correction.  These do not agree and 
## I have been unable to identify what I am doing wrong or how the correction is calculated.

In [ ]:
seqNum = 72
test = table[table.index == seqNum]
test['dof_state'].values[0]

In [ ]:
firstSeqNum = 320
lastSeqNum = 345
colors = ['black', 'red', 'green', 'blue']
fig, axes = plt.subplots(3,1,figsize=(10,15), sharex=True)
plt.subplots_adjust(hspace=0)
plt.suptitle(f"Focus Oscillations {day_obs}", fontsize=18, y=0.90)
seqNum = firstSeqNum
while seqNum <= lastSeqNum:
    for n in range(4):
        subSeq = int(seqNum + n)
        this_data = table[table.index == subSeq]
        dof_state = this_data['dof_state'].values[0]
        m2z = dof_state[0]
        camz = dof_state[5]
        zernikes_fwhm = this_data['zernikes_fwhm'].values[0]
        z4 = zernikes_fwhm[0]
        axes[0].scatter(subSeq, m2z, color = colors[n], s=60)
        axes[1].scatter(subSeq, camz, color = colors[n], s=60)
        axes[2].scatter(subSeq, z4, color=colors[n], s=60)
        if subSeq + 3 <= lastSeqNum + 1:
            delayed_data = table[table.index == subSeq + 3]
            dof_state = delayed_data['dof_state'].values[0]
            m2z = dof_state[0]
            camz = dof_state[5]
            expId = int(1E5 * day_obs + subSeq + 3)
            correction = pidDict[expId]['Correction']
            clip_i = pidDict[expId]['Clipped_I'] * 0.022
            ratio = abs(clip_i[0] / correction[0])
            #print(expId, ratio)
            axes[0].arrow(subSeq + 3, m2z, 1, -correction[0], color=colors[n], width=0.1)
            #axes[0].arrow(subSeq + 3, m2z, ratio, -clip_i[0], color=colors[n], width=0.1)
            axes[1].arrow(subSeq + 3, camz, 1, -correction[5], color=colors[n], width=0.1)

    seqNum += 4
for ax in axes:
    ax.grid(True, alpha=0.5)
axes[0].axhline(-185, ls='--', color='magenta')
axes[1].axhline(55, ls='--', color='magenta')
axes[2].axhline(-0.15, ls='--', color='magenta')
axes[0].set_ylabel("M2 Z [um]")
axes[1].set_ylabel("Cam Z [um]")
axes[2].set_ylabel("Zernikes FWHM [arcsec]")
fig.savefig(f"/home/c/cslage/u/MTAOS/times_square_reports/Focus_Oscillations_{day_obs}.png")